# ForecastOps quickstart

ForecastOps is a local-first observability and evaluation layer for production
forecasts: add one `fops.capture(...)` call after `.predict()`, and you get
Parquet artifacts, a DuckDB run index, horizon-aware metrics, validation, and a
local UI — nothing leaves your machine.

This notebook walks the full loop: **capture → metrics → validation →
benchmark → diff → report/UI**, using the synthetic daily-demand data shared by
the other examples. Run it from the `examples/` directory or the repository
root.

In [1]:
import sys
from pathlib import Path

import pandas as pd

import forecastops as fops

# Locate the examples directory whether the notebook runs from examples/ or the repo root.
EXAMPLES = Path.cwd() if (Path.cwd() / "_synthetic.py").exists() else Path.cwd() / "examples"
sys.path.insert(0, str(EXAMPLES))
STORE = EXAMPLES / ".forecastops"

from _synthetic import daily_demand, forecast_from_history  # noqa: E402

history = daily_demand(series_id="store-demo", seed=11)
forecast_df, actuals_df = forecast_from_history(history, seed=31)
forecast_df.head()

,series_id,target_time,prediction,yhat_lower,yhat_upper,region
0,store-demo,2026-04-01,110.525023,101.525023,119.525023,north
1,store-demo,2026-04-02,118.543337,109.543337,127.543337,north
2,store-demo,2026-04-03,126.561652,117.561652,135.561652,north
3,store-demo,2026-04-04,128.692617,119.692617,137.692617,north
4,store-demo,2026-04-05,123.482175,114.482175,132.482175,north


## 1. Capture a forecast

`fops.capture` is the one line you add to an existing pipeline. The
`ForecastSchema` maps whatever your dataframe columns are called onto canonical
semantics; `cutoff` marks the train/forecast boundary, and passing `actuals`
enables evaluation at capture time.

In [2]:
run = fops.capture(
    forecast_df,
    project="notebook-demo",
    schema=fops.ForecastSchema(
        series_id="series_id",
        target_time="target_time",
        prediction="prediction",
        lower="yhat_lower",
        upper="yhat_upper",
    ),
    cutoff=history["ds"].max(),
    actuals=actuals_df,
    model_name="seasonal-baseline",
    store=STORE,
)

print("run_id:", run.run_id)
print("status:", run.status)
print("artifact:", run.forecast_artifact_uri)

run_id: notebook-demo-seasonal-baseline-20260612151702-c3e7cf2aa65f
status: warning
artifact: /Users/jameslepage/Projects/castops/examples/.forecastops/artifacts/forecasts/notebook-demo-seasonal-baseline-20260612151702-c3e7cf2aa65f.parquet


## 2. Inspect metrics

Every capture with actuals computes MAE, RMSE, WAPE, bias, coverage, interval
width, and counts — overall and per horizon bucket. The records are plain
dataclasses, so they go straight into a dataframe.

In [3]:
from dataclasses import asdict

metrics = pd.DataFrame([asdict(m) for m in run.metrics])
overall = metrics[metrics.slice_name.isna() & metrics.benchmark_name.isna()]
overall[overall.horizon_bucket.isna()][["metric_name", "metric_value", "points_count"]]

,metric_name,metric_value,points_count
0,mae,2.915549,21
1,rmse,3.387152,21
2,wape,0.024140,21
3,bias,-0.432950,21
4,count,21.000000,21
5,coverage,1.000000,21
6,interval_width,18.000000,21


Error by forecast horizon — how fast does accuracy decay?

In [4]:
by_horizon = metrics[
    (metrics.slice_name == "horizon_bucket")
    & metrics.benchmark_name.isna()
    & (metrics.metric_name == "mae")
]
by_horizon[["metric_name", "horizon_bucket", "metric_value", "points_count"]]

,metric_name,horizon_bucket,metric_value,points_count
7,mae,24-48h,1.055660,1
14,mae,48h-7d,2.707992,5
21,mae,6-24h,1.581205,1
28,mae,7d+,3.217836,14


## 3. Validation

Capture also lints the forecast: schema problems, duplicate timestamps,
inverted intervals, leakage across the cutoff, and more. `run.status` rolls
these up (`ok` / `warning` / `error`).

In [5]:
validation = pd.DataFrame([asdict(event) for event in run.validation_events])
validation if not validation.empty else 'no validation issues'

,severity,code,message,affected_column,affected_count,sample,event_id
0,WARN,timezone_naive,cutoff_time or target_time is timezone-naive; ...,None,None,None,None


## 4. Benchmark a second model

Capture a second run with a naive benchmark attached. ForecastOps computes the
benchmark's metrics alongside the model's and a skill score (how much better
than the benchmark the model is).

In [6]:
forecast_v2 = forecast_df.copy()
forecast_v2["prediction"] = forecast_v2["prediction"].rolling(3, min_periods=1).mean()

benchmark = forecast_df[["series_id", "target_time"]].copy()
benchmark["yhat"] = forecast_df["prediction"].shift(1).bfill() * 1.05

run_v2 = fops.capture(
    forecast_v2,
    project="notebook-demo",
    schema=fops.ForecastSchema(
        series_id="series_id",
        target_time="target_time",
        prediction="prediction",
        lower="yhat_lower",
        upper="yhat_upper",
    ),
    cutoff=history["ds"].max(),
    actuals=actuals_df,
    benchmark=benchmark,
    benchmark_name="naive-lag1",
    model_name="seasonal-smoothed",
    store=STORE,
)

v2_metrics = pd.DataFrame([asdict(m) for m in run_v2.metrics])
skill = v2_metrics[v2_metrics.metric_name.str.startswith("skill_")]
skill[["metric_name", "metric_value", "benchmark_name", "horizon_bucket"]].head()

,metric_name,metric_value,benchmark_name,horizon_bucket
65,skill_mae,0.338228,naive-lag1,None
66,skill_rmse,0.284523,naive-lag1,None
67,skill_wape,0.338228,naive-lag1,None
68,skill_mae,-0.427625,naive-lag1,24-48h
69,skill_rmse,-0.427625,naive-lag1,24-48h


## 5. Diff two runs

`fops.diff` compares any two captured runs: per-metric deltas
(candidate − base) and flagged regressions on the error metrics.

In [7]:
result = fops.diff(run, run_v2)

print(f"regressions: {len(result.regressions)}")
result.metric_deltas[
    result.metric_deltas.slice_name.isna() & result.metric_deltas.horizon_bucket.isna()
][["metric_name", "metric_value_base", "metric_value_candidate", "delta"]]

regressions: 12


,metric_name,metric_value_base,metric_value_candidate,delta
4,bias,-0.432950,-0.550544,-0.117594
9,count,21.000000,21.000000,0.000000
14,coverage,1.000000,1.000000,0.000000
19,interval_width,18.000000,18.000000,0.000000
24,mae,2.915549,5.032925,2.117376
29,rmse,3.387152,6.357384,2.970232
34,wape,0.024140,0.041671,0.017531


## 6. Report and UI

Everything captured here landed in the local store (`examples/.forecastops`).
Generate a static HTML report for the latest run, then explore interactively:

```bash
cd examples
fops ui
```

The UI has a sortable run index, a per-series forecast inspector, project-level
error trends, and a compare view backed by the same diff you just ran.

In [8]:
report_path = fops.report(run_v2, store=STORE)
print("report written to:", report_path)

report written to: /Users/jameslepage/Projects/castops/examples/.forecastops/reports/notebook-demo-seasonal-smoothed-20260612151702-9a7677013617.html
